# 🔬 Notebook 3: Stock Exchange — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/stock-exchange
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1

### Matching engine

The matching loop:

```
while order arrives:
  if crossing the book: match against best opposite level (FIFO)
  else: rest remaining qty on book
```

**Price-time priority**: among orders at the same price, the one placed earliest matches first.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass

@dataclass
class O:
    id: int; side: str; price: float; qty: int

class Book:
    def __init__(self):
        self.bids = defaultdict(deque)   # price → FIFO of orders
        self.asks = defaultdict(deque)

    def submit(self, o):
        trades = []
        if o.side == "buy":
            # match against asks at price ≤ o.price, lowest first
            for p in sorted(self.asks):
                if p > o.price or o.qty == 0: break
                q = self.asks[p]
                while q and o.qty:
                    top = q[0]
                    fill = min(top.qty, o.qty)
                    trades.append((o.id, top.id, p, fill))
                    top.qty -= fill; o.qty -= fill
                    if top.qty == 0: q.popleft()
                if not q: del self.asks[p]
            if o.qty: self.bids[o.price].append(o)
        else:
            for p in sorted(self.bids, reverse=True):
                if p < o.price or o.qty == 0: break
                q = self.bids[p]
                while q and o.qty:
                    top = q[0]
                    fill = min(top.qty, o.qty)
                    trades.append((top.id, o.id, p, fill))
                    top.qty -= fill; o.qty -= fill
                    if top.qty == 0: q.popleft()
                if not q: del self.bids[p]
            if o.qty: self.asks[o.price].append(o)
        return trades

b = Book()
print(b.submit(O(1,"sell",101,10)))         # []
print(b.submit(O(2,"sell",102,5)))          # []
print(b.submit(O(3,"buy",101,8)))           # match 8@101
print(b.submit(O(4,"buy",103,20)))          # 2@101 + 5@102, rest rests
print("remaining bids:", {p:[(o.id,o.qty) for o in q] for p,q in b.bids.items()})

## Deep dive 2

### Market-data fan-out

One trade → thousands of subscribers. We don't send from the matcher directly (it would block). Instead, append to a ring buffer / topic and let a pool of sender threads/processes fan out.

In [ ]:
import queue, threading, time

feed = queue.Queue()
subs = [queue.Queue() for _ in range(3)]

def publisher():
    while True:
        msg = feed.get()
        if msg is None: return
        for s in subs: s.put(msg)

t = threading.Thread(target=publisher, daemon=True); t.start()

# matcher emits trades
for i in range(5):
    feed.put(f"trade#{i}")

time.sleep(0.1)
for i, s in enumerate(subs):
    out = []
    while not s.empty(): out.append(s.get())
    print(f"sub{i} got:", out)

feed.put(None)

## Closing thoughts

- Matching is a **single-threaded** problem — embrace it; use many matchers sharded by symbol.
- Persist the WAL before matching so crash recovery is just replay.
- Decouple market data fan-out from the hot path.